# Avaliação de transcrições
Compare respostas LLM e OCR com um gabarito. Use links públicos do Drive.


In [6]:
%pip -q install "jiwer>=4,<5" pandas plotly "gdown>=6,<7"

Note: you may need to restart the kernel to use updated packages.


## 1. Configuração
Informe o link do gabarito e das duas pastas de respostas. Todos devem ser públicos.


In [7]:
import json
import re
import unicodedata
from pathlib import Path

import gdown
import jiwer
import pandas as pd
import plotly.express as px
from IPython.display import display

LINK_GABARITOS = "https://drive.google.com/file/d/14VlyCmLAeg_zAHeh6wXz6SPjZ7cHDcDS/view?usp=sharing"
LINK_PASTA_LLM = "https://drive.google.com/drive/folders/1SjAmynei_WxJKISSxJ_9larKlNEcmjEq?usp=sharing"
LINK_PASTA_OCR_PIPELINE = "https://drive.google.com/drive/folders/1IJWp6P-316jIryIMVturGFRMG1gFd2QE?usp=sharing"
METRICA_ORDENACAO = "cer_sem_pontuacao"


## 2. Normalização e métricas
As funções de tratamento dos textos e o cálculo de PER permanecem iguais aos do avaliador anterior.


In [8]:
_PUNCT_CHAR = r"(?!(?<=\w)['’-](?=\w))[^\w\s]"
_PUNCT_SUB = re.compile(f"({_PUNCT_CHAR})")
_IS_PUNCT = re.compile(r"^[^\w\s]$")
PUNCT = "[PUNCT]"


def normalize_text(text):
    text = text.lower()
    text = re.sub(_PUNCT_CHAR, "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_separate_punctuation(text):
    text = text.lower()
    text = _PUNCT_SUB.sub(r" \1 ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


In [9]:
def _mask(tokens):
  return [PUNCT if _IS_PUNCT.match(t) else t for t in tokens]


def _align_punctuation(ref, hyp):
  """Alinha as sequências mascaradas e devolve os pares (i, j) de
  pontuações alinhadas entre si."""
  ref_m, hyp_m = _mask(ref), _mask(hyp)
  n, m = len(ref), len(hyp)
  big = n + m + 1

  def step(i, j):
    if ref_m[i] != hyp_m[j]:
      return big
    if ref_m[i] == PUNCT and ref[i] != hyp[j]:
      return 1
    return 0

  dist = [[0] * (m + 1) for _ in range(n + 1)]
  for i in range(1, n + 1):
    dist[i][0] = i * big
  for j in range(1, m + 1):
    dist[0][j] = j * big
  for i in range(1, n + 1):
    for j in range(1, m + 1):
        dist[i][j] = min(
          dist[i - 1][j - 1] + step(i - 1, j - 1),
          dist[i - 1][j] + big,
          dist[i][j - 1] + big,
        )

  pairs, i, j = [], n, m
  while i > 0 and j > 0:
    if dist[i][j] == dist[i - 1][j - 1] + step(i - 1, j - 1):
      if ref_m[i - 1] == PUNCT and hyp_m[j - 1] == PUNCT:
          pairs.append((i - 1, j - 1))
      i, j = i - 1, j - 1
    elif dist[i][j] == dist[i - 1][j] + big:
      i -= 1
    else:
      j -= 1
  return pairs


def punctuation_counts(reference, hypothesis):
  """Espera textos já passados por normalize_separate_punctuation."""
  ref, hyp = reference.split(), hypothesis.split()
  pairs = _align_punctuation(ref, hyp)

  n_ref = sum(1 for t in ref if _IS_PUNCT.match(t))
  n_hyp = sum(1 for t in hyp if _IS_PUNCT.match(t))
  correct = sum(ref[i] == hyp[j] for i, j in pairs)
  return {
    "C": correct,
    "S": len(pairs) - correct,
    "D": n_ref - len(pairs),
    "I": n_hyp - len(pairs),
  }


def _per(c):
  errors = c["S"] + c["D"] + c["I"]
  total = errors + c["C"]
  return errors / total if total else 0.0


def punctuation_error_rate(reference, hypothesis):
  """
  Cálculo da Punctuation Error Rate (PER), com base em https://arxiv.org/abs/2310.02943

  PER = Ip + Dp + Sp / Ip + Dp + Sp + Cp
  Ip = Pontuações Inseridas
  Dp = Pontuações Deletadas
  Sp = Pontuações Substituídas
  Cp = Pontuações Corretas
  """
  reference = normalize_separate_punctuation(reference)
  hypothesis = normalize_separate_punctuation(hypothesis)
  return _per(punctuation_counts(reference, hypothesis))


def corpus_punctuation_error_rate(references, hypotheses):
  """ PER de um corpus todo"""
  total = {"C": 0, "S": 0, "D": 0, "I": 0}
  for r, h in zip(references, hypotheses, strict=True):
    r = normalize_separate_punctuation(r)
    h = normalize_separate_punctuation(h)
    for k, v in punctuation_counts(r, h).items():
      total[k] += v
  return _per(total)


## 3. Carregar os JSONs


In [10]:
def normalize_filename(filename):
    return unicodedata.normalize("NFC", filename.replace("\\", "/").rsplit("/", 1)[-1])


pasta_entradas = Path("entradas").resolve()
pasta_entradas.mkdir(exist_ok=True)
ARQUIVO_GABARITOS = pasta_entradas / "gabarito.json"
gdown.download(LINK_GABARITOS, output=str(ARQUIVO_GABARITOS), use_cookies=False)
dados = json.loads(ARQUIVO_GABARITOS.read_text(encoding="utf-8-sig"))
gabaritos_por_arquivo = {normalize_filename(nome): texto for nome, texto in dados.items()}

fontes_respostas = []
for grupo, link in (("LLM", LINK_PASTA_LLM), ("OCR Pipeline", LINK_PASTA_OCR_PIPELINE)):
    caminhos = gdown.download_folder(link, output=str(pasta_entradas / grupo), use_cookies=False)
    for caminho in sorted(map(Path, caminhos)):
        if caminho.suffix.lower() != ".json":
            continue
        id_resposta = f"r{len(fontes_respostas) + 1:02d}"
        fontes_respostas.append({
            "id_resposta": id_resposta, "arquivo_respostas": caminho.name,
            "link_drive": link, "grupo": grupo, "caminho": caminho,
        })
        print(f"{id_resposta}: {caminho.name}")


Downloading...
From: https://drive.google.com/uc?id=14VlyCmLAeg_zAHeh6wXz6SPjZ7cHDcDS
To: C:\Users\Maicon\Desktop\8Semestre\GitHub\hackathon_ocr_historico\entradas\gabarito.json
100%|██████████| 16.1k/16.1k [00:00<00:00, 2.49MB/s]
Retrieving folder contents


Processing file 1bEtpAmFYcOkLeH_-QOAJnm8Ff7EDzfFD imagens-binarizadas-claude-sonnet-5.5-high.json
Processing file 1Ztfb-qsUJOaHJp_8k_GoU6CqUe01EOhG imagens-binarizadas-claude-sonnet-5.5-low.json
Processing file 152t97fnA1_SyU2CIAaSaCUJ0W-bwKHJD imagens-binarizadas-gemini-3.1-pro-preview-high.json
Processing file 1VTMJft_Z16E8rasRtvo0Vqb3T_a4wJF0 imagens-binarizadas-gemini-3.1-pro-preview-low.json
Processing file 1cvfJmlU5kwmXXCl2ihe1OoMCSCCoqsQ5 imagens-binarizadas-gemini-3.8-flash-high.json
Processing file 1cKn6CYY9vHETDIImEUaLxdjzfeLkFBdt imagens-binarizadas-gemini-3.8-flash-low.json
Processing file 1VueCFYkfRe4yvD0vRcAyXvNEwC2wWAxu imagens-binarizadas-gpt-5.6-terra-high.json
Processing file 1mYyf_EkdhwzUiZL3GE8rauBCLKYA-vjn imagens-binarizadas-gpt-5.6-terra-low.json
Processing file 1oKoLxwOWXj04abnvLKlVACvEIMG3X_Ai imagens-recortadas-claude-sonnet-5.5-high.json
Processing file 1ZvgcBRYVsRCsaqxtffDqfK4rwy0JVNsj imagens-recortadas-claude-sonnet-5.5-low.json
Processing file 1Im6rVPLxCQ

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1bEtpAmFYcOkLeH_-QOAJnm8Ff7EDzfFD
To: C:\Users\Maicon\Desktop\8Semestre\GitHub\hackathon_ocr_historico\entradas\LLM\imagens-binarizadas-claude-sonnet-5.5-high.json
100%|██████████| 18.7k/18.7k [00:00<00:00, 1.42MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Ztfb-qsUJOaHJp_8k_GoU6CqUe01EOhG
To: C:\Users\Maicon\Desktop\8Semestre\GitHub\hackathon_ocr_historico\entradas\LLM\imagens-binarizadas-claude-sonnet-5.5-low.json
100%|██████████| 18.7k/18.7k [00:00<00:00, 1.31MB/s]
Downloading...
From: https://drive.google.com/uc?id=152t97fnA1_SyU2CIAaSaCUJ0W-bwKHJD
To: C:\Users\Maicon\Desktop\8Semestre\GitHub\hackathon_ocr_historico\entradas\LLM\imagens-binarizadas-gemini-3.1-pro-preview-high.json
100%|██████████| 18.8k/18.8k [00:00<00:00, 473kB/s]
Downloading...
From: https://drive.google.com/uc?id=1VTMJft_Z16E8rasRtvo0Vqb3T_a4wJF0

r01: imagens-binarizadas-claude-sonnet-5.5-high.json
r02: imagens-binarizadas-claude-sonnet-5.5-low.json
r03: imagens-binarizadas-gemini-3.1-pro-preview-high.json
r04: imagens-binarizadas-gemini-3.1-pro-preview-low.json
r05: imagens-binarizadas-gemini-3.8-flash-high.json
r06: imagens-binarizadas-gemini-3.8-flash-low.json
r07: imagens-binarizadas-gpt-5.6-terra-high.json
r08: imagens-binarizadas-gpt-5.6-terra-low.json
r09: imagens-recortadas-claude-sonnet-5.5-high.json
r10: imagens-recortadas-claude-sonnet-5.5-low.json
r11: imagens-recortadas-gemini-3.1-pro-preview-high.json
r12: imagens-recortadas-gemini-3.1-pro-preview-low.json
r13: imagens-recortadas-gemini-3.8-flash-high.json
r14: imagens-recortadas-gemini-3.8-flash-low.json
r15: imagens-recortadas-gpt-5.6-terra-high.json
r16: imagens-recortadas-gpt-5.6-terra-low.json
Processing file 1QYRo_51DeegAik6gspXR1cYEg8kZSmc6 kraken_binarizada.json
Processing file 1Ux94ZvPhHKHewCSfn0sZia3Q_uUvsxBU kraken_binarizada_2x.json
Processing file 1Fa

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1QYRo_51DeegAik6gspXR1cYEg8kZSmc6
To: C:\Users\Maicon\Desktop\8Semestre\GitHub\hackathon_ocr_historico\entradas\OCR Pipeline\kraken_binarizada.json
100%|██████████| 14.8k/14.8k [00:00<00:00, 1.18MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Ux94ZvPhHKHewCSfn0sZia3Q_uUvsxBU
To: C:\Users\Maicon\Desktop\8Semestre\GitHub\hackathon_ocr_historico\entradas\OCR Pipeline\kraken_binarizada_2x.json
100%|██████████| 15.4k/15.4k [00:00<00:00, 517kB/s]
Downloading...
From: https://drive.google.com/uc?id=1FaOBJtIENTLJhfZbeycauPEda2T078GK
To: C:\Users\Maicon\Desktop\8Semestre\GitHub\hackathon_ocr_historico\entradas\OCR Pipeline\kraken_nlbin.json
100%|██████████| 14.8k/14.8k [00:00<00:00, 5.76MB/s]
Downloading...
From: https://drive.google.com/uc?id=1YU86HXoOpy7eu7mkRuSUnQ676POOnvmi
To: C:\Users\Maicon\Desktop\8Semestre\GitHub\hackatho

r17: kraken_binarizada.json
r18: kraken_binarizada_2x.json
r19: kraken_nlbin.json
r20: kraken_original.json
r21: kraken_original_2x.json



Download completed


## 4. Avaliar


In [11]:
METRICAS = [
    "wer_sem_pontuacao", "cer_sem_pontuacao",
    "wer_com_pontuacao", "cer_com_pontuacao", "per",
]
COLUNAS_CONTEXTO = ["id_resposta", "arquivo_respostas", "link_drive", "grupo", "modelo", "esforco_raciocinio", "entrada"]


def calcular_metricas(referencias, hipoteses):
    if not referencias:
        return dict.fromkeys(METRICAS)
    refs_sem = [normalize_text(t) for t in referencias]
    hyps_sem = [normalize_text(t) for t in hipoteses]
    refs_com = [normalize_separate_punctuation(t) for t in referencias]
    hyps_com = [normalize_separate_punctuation(t) for t in hipoteses]
    return {
        "wer_sem_pontuacao": jiwer.wer(refs_sem, hyps_sem),
        "cer_sem_pontuacao": jiwer.cer(refs_sem, hyps_sem),
        "wer_com_pontuacao": jiwer.wer(refs_com, hyps_com),
        "cer_com_pontuacao": jiwer.cer(refs_com, hyps_com),
        "per": corpus_punctuation_error_rate(referencias, hipoteses),
    }


def metadados_execucao(resultados, arquivo_respostas):
    modelo_curto = resultados["modelo"].rsplit("/", 1)[-1]
    nome = Path(arquivo_respostas).stem
    entrada, separador, sufixo = nome.rpartition(f"-{modelo_curto}")
    niveis = {"none", "minimal", "low", "medium", "high", "xhigh", "max", "auto"}
    esforco_nome = sufixo.lstrip("-") if separador else ""
    esforco_nome = esforco_nome if esforco_nome in niveis else None
    esforco_json = resultados.get("esforco_raciocinio")
    if esforco_json and esforco_nome and esforco_json != esforco_nome:
        print(f"{arquivo_respostas}: esforco do nome ({esforco_nome}) difere do JSON ({esforco_json}); usando JSON.")
    return {
        "esforco_raciocinio": esforco_json or esforco_nome or "não informado",
        "entrada": entrada if separador else resultados.get("entrada", nome),
    }


def avaliar_resposta(fonte, gabaritos):
    resultados = json.loads(fonte["caminho"].read_text(encoding="utf-8-sig"))
    contexto = {k: fonte[k] for k in COLUNAS_CONTEXTO if k in fonte}
    contexto["modelo"] = resultados["modelo"]
    contexto.update(metadados_execucao(resultados, fonte["arquivo_respostas"]))
    linhas, referencias, hipoteses = [], [], []
    for imagem in resultados["imagens"]:
        nome = normalize_filename(imagem["arquivo"])
        referencia, transcricao = gabaritos[nome], imagem["texto_transcrito"]
        linhas.append({**contexto, "arquivo": nome, **calcular_metricas([referencia], [transcricao])})
        referencias.append(referencia)
        hipoteses.append(transcricao)
    resumo = {**contexto, **calcular_metricas(referencias, hipoteses)}
    return {"linhas": linhas, "resumo": resumo}


avaliacoes = [avaliar_resposta(fonte, gabaritos_por_arquivo) for fonte in fontes_respostas]
todas_linhas = [linha for avaliacao in avaliacoes for linha in avaliacao["linhas"]]
resumos = [avaliacao["resumo"] for avaliacao in avaliacoes]


## 5. Comparar


In [12]:
nomes_imagens = [linha["arquivo"] for linha in avaliacoes[0]["linhas"]]
df_resumo = pd.DataFrame(resumos)
df_imagens = pd.DataFrame(todas_linhas)
df_ranking = df_resumo.sort_values([METRICA_ORDENACAO, "id_resposta"]).copy()
def rotulo_execucao(resumo):
    if resumo["grupo"] == "OCR Pipeline":
        descricao = f"{resumo['modelo']} ({Path(resumo['arquivo_respostas']).stem})"
    else:
        modelo_curto = resumo["modelo"].rsplit("/", 1)[-1]
        descricao = f"{modelo_curto} ({resumo['entrada']}; esforço: {resumo['esforco_raciocinio']})"
    return f"{resumo['id_resposta']} - {descricao}"


rotulos = {r["id_resposta"]: rotulo_execucao(r) for r in resumos}
df_ranking["rotulo"] = df_ranking["id_resposta"].map(rotulos)


## 6. Ranking
Erros em porcentagem. Valores menores indicam melhor desempenho.


In [13]:
NOMES_METRICAS = {
    "wer_sem_pontuacao": "WER s/ pont.", "cer_sem_pontuacao": "CER s/ pont.",
    "wer_com_pontuacao": "WER c/ pont.", "cer_com_pontuacao": "CER c/ pont.", "per": "PER",
}
NOMES_COLUNAS = {
    "id_resposta": "ID", "grupo": "Grupo", "modelo": "Modelo", "arquivo_respostas": "Arquivo JSON",
    "esforco_raciocinio": "Esforço de raciocínio", "entrada": "Entrada",
    **NOMES_METRICAS,
}
CORES_GRUPOS = {"LLM": "#2563eb", "OCR Pipeline": "#ea580c"}
ALTURA_GRAFICOS = max(360, 55 * len(avaliacoes) + 160)


def mostrar_tabela(df, colunas):
    tabela = df[colunas].copy()
    for coluna in METRICAS:
        if coluna in tabela:
            tabela[coluna] = tabela[coluna].map(lambda v: "—" if pd.isna(v) else f"{v:.2%}")
    tabela = tabela.rename(columns=NOMES_COLUNAS).fillna("—").set_index("ID")
    with pd.option_context("display.max_columns", None, "display.max_colwidth", 45):
        display(tabela)


def mostrar_grafico(figura, titulo):
    figura.update_layout(
        title=titulo, template="plotly_white", height=ALTURA_GRAFICOS,
        margin=dict(l=20, r=20, t=70, b=60), font=dict(size=12),
        legend=dict(orientation="h", y=-0.2, x=0),
    )
    figura.update_yaxes(autorange="reversed", automargin=True)
    figura.show(config={"displaylogo": False, "responsive": True})


mostrar_tabela(df_ranking, ["id_resposta", "grupo", "modelo", "esforco_raciocinio", "entrada", *METRICAS])


,Grupo,Modelo,Esforço de raciocínio,Entrada,WER s/ pont.,CER s/ pont.,WER c/ pont.,CER c/ pont.,PER
ID,,,,,,,,,
r14,LLM,google/gemini-3.8-flash,low,imagens-recortadas,12.33%,4.31%,17.06%,5.65%,25.99%
r12,LLM,google/gemini-3.1-pro-preview,low,imagens-recortadas,14.01%,4.40%,15.59%,5.13%,22.11%
r13,LLM,google/gemini-3.8-flash,high,imagens-recortadas,13.29%,4.44%,20.34%,6.13%,29.57%
r11,LLM,google/gemini-3.1-pro-preview,high,imagens-recortadas,13.13%,4.55%,15.92%,5.45%,23.37%
r03,LLM,google/gemini-3.1-pro-preview,high,imagens-binarizadas,13.34%,4.61%,17.83%,5.95%,30.61%
r05,LLM,google/gemini-3.8-flash,high,imagens-binarizadas,13.42%,4.92%,20.72%,6.90%,34.70%
r04,LLM,google/gemini-3.1-pro-preview,low,imagens-binarizadas,15.39%,5.29%,17.39%,6.23%,27.34%
r09,LLM,anthropic/claude-sonnet-5.5,high,imagens-recortadas,15.31%,5.57%,16.54%,6.29%,22.94%
r06,LLM,google/gemini-3.8-flash,low,imagens-binarizadas,14.64%,5.58%,22.11%,7.56%,37.08%


In [14]:
dados = df_ranking.assign(taxa_percentual=df_ranking[METRICA_ORDENACAO] * 100)
fig_ranking = px.bar(
    dados, x="taxa_percentual", y="rotulo", color="grupo", orientation="h",
    color_discrete_map=CORES_GRUPOS, text="taxa_percentual",
    hover_data=["modelo", "esforco_raciocinio", "entrada", "arquivo_respostas"],
    labels={"taxa_percentual": f"{NOMES_METRICAS[METRICA_ORDENACAO]} (%)", "rotulo": "Execução", "grupo": "Grupo"},
)
fig_ranking.update_traces(texttemplate="%{x:.2f}%", textposition="outside", cliponaxis=False)
fig_ranking.update_xaxes(range=[0, max(1, dados["taxa_percentual"].max() * 1.2)])
fig_ranking.update_yaxes(categoryorder="array", categoryarray=dados["rotulo"].tolist())
mostrar_grafico(fig_ranking, "Ranking")

fig_metricas = px.imshow(
    df_ranking[METRICAS].to_numpy(dtype=float) * 100,
    x=[NOMES_METRICAS[m] for m in METRICAS], y=df_ranking["rotulo"].tolist(),
    color_continuous_scale="YlOrRd", zmin=0, aspect="auto", text_auto=".2f",
    labels={"x": "Métrica", "y": "Execução", "color": "Erro (%)"},
)
fig_metricas.update_traces(hovertemplate="%{y}<br>%{x}: %{z:.2f}%<extra></extra>")
mostrar_grafico(fig_metricas, "Métricas (%)")


## 7. Respostas


In [15]:
mostrar_tabela(df_resumo, [
    "id_resposta", "grupo", "arquivo_respostas", "modelo", "esforco_raciocinio", "entrada",
])


,Grupo,Arquivo JSON,Modelo,Esforço de raciocínio,Entrada
ID,,,,,
r01,LLM,imagens-binarizadas-claude-sonnet-5.5-hig...,anthropic/claude-sonnet-5.5,high,imagens-binarizadas
r02,LLM,imagens-binarizadas-claude-sonnet-5.5-low...,anthropic/claude-sonnet-5.5,low,imagens-binarizadas
r03,LLM,imagens-binarizadas-gemini-3.1-pro-previe...,google/gemini-3.1-pro-preview,high,imagens-binarizadas
r04,LLM,imagens-binarizadas-gemini-3.1-pro-previe...,google/gemini-3.1-pro-preview,low,imagens-binarizadas
r05,LLM,imagens-binarizadas-gemini-3.8-flash-high...,google/gemini-3.8-flash,high,imagens-binarizadas
r06,LLM,imagens-binarizadas-gemini-3.8-flash-low....,google/gemini-3.8-flash,low,imagens-binarizadas
r07,LLM,imagens-binarizadas-gpt-5.6-terra-high.json,openai/gpt-5.6-terra,high,imagens-binarizadas
r08,LLM,imagens-binarizadas-gpt-5.6-terra-low.json,openai/gpt-5.6-terra,low,imagens-binarizadas
r09,LLM,imagens-recortadas-claude-sonnet-5.5-high...,anthropic/claude-sonnet-5.5,high,imagens-recortadas


## 8. Resultados por imagem


In [16]:
matriz = df_imagens.pivot(index="id_resposta", columns="arquivo", values="cer_sem_pontuacao")
matriz = matriz.reindex(index=df_ranking["id_resposta"], columns=nomes_imagens)
fig_imagens = px.imshow(
    matriz.to_numpy(dtype=float) * 100,
    x=[f"Imagem {i}" for i in range(1, len(nomes_imagens) + 1)],
    y=[rotulos[id_resposta] for id_resposta in matriz.index],
    color_continuous_scale="YlOrRd", zmin=0, aspect="auto",
    labels={"x": "Documento", "y": "Execução", "color": "CER (%)"},
)
fig_imagens.update_traces(
    customdata=[nomes_imagens for _ in matriz.index],
    hovertemplate="%{y}<br>%{customdata}<br>CER: %{z:.2f}%<extra></extra>",
)
mostrar_grafico(fig_imagens, "CER sem pontuação por imagem")


## 9. Custo e custo-benefício dos LLMs
Custos em US$, somados do campo `custo` de cada imagem. Cada barra representa uma execução (modelo e tipo de entrada); o hover mostra também o custo médio por imagem. A comparação usa o mesmo conjunto de documentos e exclui execuções com custos ausentes ou inválidos. OCR Pipeline não participa.

A nota combina **acerto estimado = max(0, 1 − CER sem pontuação)** e **economia = menor custo total / custo total da execução**. A fórmula é `100 × (PESO_ACERTO × acerto + (1 − PESO_ACERTO) × economia)`. O peso inicial é 50% para cada componente e pode ser alterado abaixo. A economia é relativa às execuções comparadas; custo zero recebe economia 1. Esta é uma nota de comparação, e o acerto é uma aproximação derivada do CER.


In [17]:
import math

# 0 = apenas economia; 1 = apenas acerto.
PESO_ACERTO = 0.5
if not 0 <= PESO_ACERTO <= 1:
    raise ValueError("PESO_ACERTO deve estar entre 0 e 1.")

resumos_por_id = {r["id_resposta"]: r for r in resumos}
linhas_custos_llm = []
for fonte in fontes_respostas:
    if fonte["grupo"] != "LLM":
        continue
    resultado = json.loads(fonte["caminho"].read_text(encoding="utf-8-sig"))
    imagens = resultado["imagens"]
    custos = [imagem.get("custo") for imagem in imagens]
    custos_validos = bool(custos) and all(
        isinstance(custo, (int, float)) and not isinstance(custo, bool)
        and math.isfinite(custo) and custo >= 0
        for custo in custos
    )
    arquivos = [normalize_filename(imagem["arquivo"]) for imagem in imagens]
    mesmos_documentos = len(arquivos) == len(set(arquivos)) and set(arquivos) == set(nomes_imagens)
    if not custos_validos or not mesmos_documentos:
        motivo = "custo ausente/inválido" if not custos_validos else "conjunto de documentos diferente"
        print(f"{fonte['id_resposta']} fora da comparação de custos: {motivo}.")
        continue
    linhas_custos_llm.append({
        "id_resposta": fonte["id_resposta"],
        "rotulo_custo": rotulo_execucao(resumos_por_id[fonte["id_resposta"]]),
        "custo_total_usd": sum(custos),
        "custo_por_imagem_usd": sum(custos) / len(custos),
        "n_imagens": len(custos),
    })

if not linhas_custos_llm:
    raise ValueError("Nenhuma execução LLM com custos completos para os mesmos documentos.")

df_custos_llm = df_resumo.loc[df_resumo["grupo"].eq("LLM")].merge(
    pd.DataFrame(linhas_custos_llm), on="id_resposta", validate="one_to_one",
)
# Proxy de acerto, limitada a [0, 1], pois CER pode ultrapassar 100%.
df_custos_llm["acerto"] = (1 - df_custos_llm["cer_sem_pontuacao"]).clip(0, 1)
menor_custo = df_custos_llm["custo_total_usd"].min()
df_custos_llm["economia"] = df_custos_llm["custo_total_usd"].map(
    lambda custo: 1.0 if custo == 0 else menor_custo / custo
)
df_custos_llm["acerto_percentual"] = df_custos_llm["acerto"] * 100
df_custos_llm["economia_percentual"] = df_custos_llm["economia"] * 100
df_custos_llm["nota_custo_beneficio"] = 100 * (
    PESO_ACERTO * df_custos_llm["acerto"]
    + (1 - PESO_ACERTO) * df_custos_llm["economia"]
)

ROTULOS_CUSTOS = {
    "esforco_raciocinio": "Esforço de raciocínio", "entrada": "Entrada",
    "rotulo_custo": "Execução LLM", "modelo": "Modelo",
    "custo_total_usd": "Custo total (US$)",
    "custo_por_imagem_usd": "Custo por imagem (US$)", "n_imagens": "Imagens",
    "acerto_percentual": "Acerto estimado (%)", "economia_percentual": "Economia relativa (%)",
    "nota_custo_beneficio": "Nota de custo-benefício (0–100)",
}


r13 fora da comparação de custos: conjunto de documentos diferente.


In [18]:
dados_custos = df_custos_llm.sort_values(["custo_total_usd", "id_resposta"])
fig_custos_llm = px.bar(
    dados_custos, x="custo_total_usd", y="rotulo_custo", color="modelo", orientation="h",
    text="custo_total_usd", labels=ROTULOS_CUSTOS,
    hover_data={"esforco_raciocinio": True, "entrada": True, "custo_por_imagem_usd": ":.6f", "n_imagens": True, "arquivo_respostas": True},
)
fig_custos_llm.update_traces(texttemplate="US$ %{x:.4f}", textposition="outside", cliponaxis=False)
fig_custos_llm.update_xaxes(range=[0, max(0.01, dados_custos["custo_total_usd"].max() * 1.25)])
fig_custos_llm.update_yaxes(categoryorder="array", categoryarray=dados_custos["rotulo_custo"].tolist())
mostrar_grafico(fig_custos_llm, "Custo total por execução LLM — menor é melhor")


In [19]:
dados_beneficio = df_custos_llm.sort_values(
    ["nota_custo_beneficio", "id_resposta"], ascending=[False, True],
)
fig_custo_beneficio = px.bar(
    dados_beneficio, x="nota_custo_beneficio", y="rotulo_custo", color="modelo", orientation="h",
    text="nota_custo_beneficio", labels=ROTULOS_CUSTOS,
    hover_data={"esforco_raciocinio": True, "entrada": True, 
        "acerto_percentual": ":.2f", "economia_percentual": ":.2f",
        "custo_total_usd": ":.6f", "custo_por_imagem_usd": ":.6f", "arquivo_respostas": True,
    },
)
fig_custo_beneficio.update_traces(texttemplate="%{x:.2f}", textposition="outside", cliponaxis=False)
fig_custo_beneficio.update_xaxes(range=[0, 110])
fig_custo_beneficio.update_yaxes(categoryorder="array", categoryarray=dados_beneficio["rotulo_custo"].tolist())
mostrar_grafico(
    fig_custo_beneficio,
    f"Custo-benefício LLM — maior é melhor (acerto {PESO_ACERTO:.0%} / economia {1 - PESO_ACERTO:.0%})",
)


## 10. Acerto × custo total dos LLMs
Cada ponto representa uma execução LLM. O eixo vertical mostra o **acerto estimado (%) = 100 × max(0, 1 − CER sem pontuação)**, já usado na nota de custo-benefício. O eixo horizontal mostra o **custo final em US$**, somando todas as imagens da execução, sem normalização pela nota de economia ou projeção por milhão de páginas.

Os rótulos mostram o ID de cada execução. As cores identificam os modelos e os símbolos identificam o esforço de raciocínio. Passe o mouse para ver a configuração completa, o custo total e o número de imagens. Pontos mais altos e à esquerda combinam maior acerto e menor custo. O eixo de custo usa escala linear, começando em zero: distâncias iguais representam diferenças iguais em dólares. São mantidas as mesmas execuções elegíveis da comparação de custos da seção anterior.


In [20]:
dados_acerto_custo = df_custos_llm.sort_values(
    ["modelo", "esforco_raciocinio", "entrada", "id_resposta"],
).copy()
# IDs curtos mantêm os rótulos legíveis; o hover mostra a configuração completa.
dados_acerto_custo["rotulo_ponto"] = dados_acerto_custo["id_resposta"]
fig_acerto_custo_llm = px.scatter(
    dados_acerto_custo,
    x="custo_total_usd", y="acerto_percentual",
    color="modelo", symbol="esforco_raciocinio", text="rotulo_ponto",
    log_x=False, labels=ROTULOS_CUSTOS,
    hover_name="rotulo_custo",
    hover_data={
        "rotulo_ponto": False, "id_resposta": True, "entrada": True,
        "acerto_percentual": ":.2f", "custo_total_usd": ":.6f",
        "custo_por_imagem_usd": ":.6f", "n_imagens": True,
        "arquivo_respostas": True,
    },
)
fig_acerto_custo_llm.update_traces(
    marker=dict(size=13, line=dict(width=1, color="white")),
    textposition="top center", textfont=dict(size=11), cliponaxis=False,
)
fig_acerto_custo_llm.update_layout(
    title="LLMs: acerto estimado × custo total da execução",
    template="plotly_white", height=700,
    margin=dict(l=70, r=70, t=90, b=160), font=dict(size=12),
    legend=dict(orientation="h", y=-0.22, x=0, title_text="Modelo / esforço de raciocínio"),
)
fig_acerto_custo_llm.update_xaxes(
    title_text="Custo total da execução (US$)", type="linear", rangemode="tozero",
    tickprefix="US$ ", tickformat=".3f", automargin=True,
)
# Não usa mostrar_grafico(), pois esse helper inverte o eixo Y dos rankings.
limites_acerto = [
    max(0, dados_acerto_custo["acerto_percentual"].min() - 5),
    min(105, dados_acerto_custo["acerto_percentual"].max() + 5),
]
fig_acerto_custo_llm.update_yaxes(
    title_text="Acerto estimado (%)", range=limites_acerto,
    autorange=False, ticksuffix="%", automargin=True,
)
fig_acerto_custo_llm.show(config={"displaylogo": False, "responsive": True})
